In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import time

print("Libraries loaded.")
print("MPS available:", torch.backends.mps.is_available())

Libraries loaded.
MPS available: True


In [2]:
X_train = np.load('../data/processed/X_train_resampled.npy')
y_train = pd.read_csv('../data/processed/y_train_resampled.csv').squeeze()

X_val = np.load('../data/processed/X_val_scaled.npy')
y_val = pd.read_csv('../data/processed/y_val_clean.csv').squeeze()

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

X_train shape: (2062829, 71)
X_val shape: (423051, 71)


In [3]:
class NetworkFlowDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [4]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

train_dataset = NetworkFlowDataset(X_train, y_train_encoded)
val_dataset = NetworkFlowDataset(X_val, y_val_encoded)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Training batches: 4029
Validation batches: 827


In [5]:
class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(BiLSTM, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )
        
        self.fc = nn.Linear(hidden_size * 2, num_classes)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        x = x.unsqueeze(2)
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]
        out = self.dropout(out)
        out = self.fc(out)
        return out

In [6]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = BiLSTM(
    input_size=1,
    hidden_size=128,
    num_layers=2,
    num_classes=15
).to(device)

print(f"Device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Device: mps
Model parameters: 533,263


In [7]:
#loss parameter
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Loss function: CrossEntropyLoss")
print("Optimizer: Adam, lr=0.001")

Loss function: CrossEntropyLoss
Optimizer: Adam, lr=0.001


In [8]:
print("Training Started")
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10):
    model.train()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
        
        train_acc = 100 * correct / total
        train_loss = running_loss / len(train_loader)
        
        print(f"Epoch {epoch+1}/{epochs} — Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")

train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10)

Training Started
Epoch 1/10 — Loss: 0.2159, Accuracy: 93.16%
Epoch 2/10 — Loss: 0.1005, Accuracy: 95.89%
Epoch 3/10 — Loss: 0.0881, Accuracy: 96.28%
Epoch 4/10 — Loss: 0.0816, Accuracy: 96.49%
Epoch 5/10 — Loss: 0.0783, Accuracy: 96.62%
Epoch 6/10 — Loss: 0.0729, Accuracy: 96.89%
Epoch 7/10 — Loss: 0.0717, Accuracy: 96.94%
Epoch 8/10 — Loss: 0.0687, Accuracy: 97.04%
Epoch 9/10 — Loss: 0.0664, Accuracy: 97.15%
Epoch 10/10 — Loss: 0.0650, Accuracy: 97.19%


In [13]:
torch.save(model.state_dict(), '../models/bilstm.pth')
print("Model saved.")

Model saved.


In [14]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in val_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        
        outputs = model(batch_X)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

all_preds = le.inverse_transform(all_preds)
all_labels = le.inverse_transform(all_labels)

print(classification_report(all_labels, all_preds, digits=4))

                            precision    recall  f1-score   support

                    BENIGN     0.9940    0.9840    0.9889    339790
                       Bot     0.5774    0.6621    0.6169       293
                      DDoS     0.9930    0.9812    0.9870     19153
             DoS GoldenEye     0.9736    0.9584    0.9660      1540
                  DoS Hulk     0.9687    0.9874    0.9779     34427
          DoS Slowhttptest     0.8948    0.9818    0.9363       823
             DoS slowloris     0.9456    0.9815    0.9632       867
               FTP-Patator     0.9741    0.9806    0.9773      1187
                Heartbleed     0.6667    1.0000    0.8000         2
              Infiltration     0.0938    0.6000    0.1622         5
                  PortScan     0.8955    0.9545    0.9241     23757
               SSH-Patator     0.9277    0.8288    0.8754       882
  Web Attack - Brute Force     0.4894    0.1022    0.1691       225
Web Attack - Sql Injection     0.0051    0.3333